In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from datetime import timedelta, date

In [2]:
pd.set_option('max_colwidth', 800)

## Initial data load & cleaning

In [3]:
data_load = pd.read_csv('../data/trash_hauler_report_with_lat_lng.csv')

In [4]:
data_load

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
0,25270,11/01/17,Trash - Backdoor,"house with the wheel chair ramp, they share driveway, in back driveway. 3817 Crouch Dr near NES light post is. \n615-876-6274",3817 Crouch Dr,37207.0,RED RIVER,3205,2.0,1.727970e+06,686779.478089,-86.815392,36.217292
1,25274,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1.721259e+06,685444.799565,-86.838103,36.213470
2,25276,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1.707027e+06,659887.471571,-86.885562,36.142923
3,25307,11/01/17,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1.735692e+06,685027.245923,-86.789170,36.212652
4,25312,11/01/17,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1.710186e+06,664205.101066,-86.874995,36.154861
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20221,267125,11/01/19,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1.781137e+06,632448.551144,-86.633970,36.069130
20222,267126,11/01/19,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1.749711e+06,669201.601569,-86.741242,36.169482
20223,267130,11/01/19,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, United States",37214.0,RED RIVER,1502,15.0,1.770293e+06,674936.303809,-86.671647,36.185643
20224,267134,11/01/19,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only picked up 3x in the last 3 month.,"3325 Murfreesboro Pike, Nashville, TN 37013, United States",37013.0,RED RIVER,4502,32.0,1.785225e+06,627146.400187,-86.620025,36.054637


In [5]:
data_load.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20226 entries, 0 to 20225
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Request Number    20226 non-null  int64  
 1   Date Opened       20226 non-null  object 
 2   Request           20226 non-null  object 
 3   Description       20195 non-null  object 
 4   Incident Address  20217 non-null  object 
 5   Zip Code          20151 non-null  float64
 6   Trash Hauler      19325 non-null  object 
 7   Trash Route       19279 non-null  object 
 8   Council District  20177 non-null  float64
 9   State Plan X      20198 non-null  float64
 10  State Plan Y      20198 non-null  float64
 11  LONGITUDE         20198 non-null  float64
 12  LATITUDE          20198 non-null  float64
dtypes: float64(6), int64(1), object(6)
memory usage: 2.0+ MB


In [6]:
data_load['Request '].value_counts()

Request 
Trash - Curbside/Alley Missed Pickup    15028
Trash - Backdoor                         2629
Trash Collection Complaint               2312
Damage to Property                        257
Name: count, dtype: int64

#### First I want to find all the rows that refer to missed pickups.

A quick look through the different request types (below) indicates that merely filtering for 'Curbside/Alley Missed Pickup' won't suffice, as there are calls about missed pickups in the 'Backdoor' and 'Trash Collection Complaint' categories as well.  I don't see any under 'Damage to Property', but will look for them just in case.

In [7]:
data_load.loc[data_load['Request '] == 'Trash - Curbside/Alley Missed Pickup', 'Description']

1                                                            Curb/Trash miss Tuesday.
2                                                            Curb/trash miss Tuesday.
3                                                                              missed
4                                                   Missed the even side of the road.
8                                                                             Missed.
                                             ...                                     
20221                                                       MISSED...NEIGHBORS MISSED
20222                                                                    entire alley
20223                                                                  missed several
20224    Caller stated trash was missed & were only picked up 3x in the last 3 month.
20225                                                  possibly others missed as well
Name: Description, Length: 15028, dtype: object

In [8]:
data_load.loc[data_load['Request '] == 'Trash - Backdoor', 'Description']

0        house with the wheel chair ramp, they share driveway, in back driveway. 3817 Crouch Dr near NES light post is. \n615-876-6274
55                                                                             Backdoor/miss for last Friday. Does not understand why?
63                                          Missed trash pickup said has been picking up behind the gate for 26 years \n(615) 333-0065
71                                                                                                                             Missed.
78                                Missed- she said her grandson was over yesterday and he told her it was full. She needs it picked up
                                                                     ...                                                              
20165                                                         HAS MISSED BACK DOOR TRASH PICK UP AGAIN/ ALSO MISSED LAST WEEKS PICK UP
20167                                                  

In [9]:
data_load.loc[data_load['Request '] == 'Trash Collection Complaint', 'Description']

5                                                                                                                                                                                                                                                           left trash cart in middle of driveway instead of at the backdoor pickup spot
7                                                                                                                                                                                                                                                                                           Trash out on time, miss again Tuesday. ALLEY
11                                                                                                                                                                                                                                                                                                            Missed- 4th week in a row.
13           

In [10]:
data_load.loc[data_load['Request '] == 'Damage to Property', 'Description']

6                                                                                                                  Trash/emptied Wednesday & now metal black-mailbox damage. \ncustomer wants call back and fix and replace.
173                                                                                                                                                                             truck is cutting into yard and damaging lawn
257                                                                                               cable lines pulled from house - caused damage to roof has pictures\ncomcast on site repair line itself but roof has damage
360      Customer does not understands why trash-truck bent mail-box again today. Damage mail-box won't close mail will get wet if it rains. {Repair/Please} 2nd time if any questions call her. \n615-876-9325\nJudy Cooper
384                                                                                                                 

### Filter for missed pickups
To analyze the appropriate requests, I will filter the data for rows where the request is 'Missed Pickup' or the description contains either miss/missed/missing or 'no pick up'/'not picked up'

In [11]:
data_load['Description'] = data_load['Description'].fillna('[No description]')

In [12]:
missed_pickups = data_load.loc[
    (data_load['Request '].str.contains('Missed')) | 
    ## any capitalization of 'miss'
    (data_load['Description'].str.contains(r'\b[mM][iI][sS]{2}.?', regex = True)) | 
    ## any capitalization of 'no pickup'/'no pick up'/'not picked up'
    (data_load['Description'].str.contains(r'\b[nN][oO].*\b[pP][iI][cC][kK].*[uU][pP]', regex = True))].reset_index(drop = True)

In [13]:
missed_pickups

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
0,25274,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1.721259e+06,685444.799565,-86.838103,36.213470
1,25276,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1.707027e+06,659887.471571,-86.885562,36.142923
2,25307,11/01/17,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1.735692e+06,685027.245923,-86.789170,36.212652
3,25312,11/01/17,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1.710186e+06,664205.101066,-86.874995,36.154861
4,25327,11/01/17,Trash Collection Complaint,"Trash out on time, miss again Tuesday. ALLEY",1816 Jo Johnston Ave,37203.0,METRO,9208,21.0,1.731459e+06,666013.601229,-86.802988,36.160330
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18164,267125,11/01/19,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1.781137e+06,632448.551144,-86.633970,36.069130
18165,267126,11/01/19,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1.749711e+06,669201.601569,-86.741242,36.169482
18166,267130,11/01/19,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, United States",37214.0,RED RIVER,1502,15.0,1.770293e+06,674936.303809,-86.671647,36.185643
18167,267134,11/01/19,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only picked up 3x in the last 3 month.,"3325 Murfreesboro Pike, Nashville, TN 37013, United States",37013.0,RED RIVER,4502,32.0,1.785225e+06,627146.400187,-86.620025,36.054637


### Normalize 'Trash Hauler'

In [14]:
missed_pickups['Trash Hauler'] = missed_pickups['Trash Hauler'].str.title()

In [15]:
missed_pickups['Trash Hauler'].value_counts(dropna = False)

Trash Hauler
Red River    13125
Metro         3098
Waste Ind     1168
NaN            778
Name: count, dtype: int64

In [16]:
missed_pickups.loc[missed_pickups['Trash Hauler'].isna()].info() 

<class 'pandas.core.frame.DataFrame'>
Index: 778 entries, 73 to 18155
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Request Number    778 non-null    int64  
 1   Date Opened       778 non-null    object 
 2   Request           778 non-null    object 
 3   Description       778 non-null    object 
 4   Incident Address  771 non-null    object 
 5   Zip Code          753 non-null    float64
 6   Trash Hauler      0 non-null      object 
 7   Trash Route       33 non-null     object 
 8   Council District  755 non-null    float64
 9   State Plan X      760 non-null    float64
 10  State Plan Y      760 non-null    float64
 11  LONGITUDE         760 non-null    float64
 12  LATITUDE          760 non-null    float64
dtypes: float64(6), int64(1), object(6)
memory usage: 85.1+ KB


In [17]:
missed_pickups['Trash Hauler'] = missed_pickups['Trash Hauler'].fillna('No hauler identified')

#### Can we further investigate those nulls and infer the hauler?

### Normalize addresses

In [18]:
missed_pickups.loc[missed_pickups['Incident Address'].isna()]

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
539,33128,12/13/17,Trash - Curbside/Alley Missed Pickup,Missed.,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
761,35857,12/29/17,Trash - Curbside/Alley Missed Pickup,daughters car parked in front,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
1017,39689,01/17/18,Trash - Curbside/Alley Missed Pickup,Trash pick up not done for Tuesday 1/16/18,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
1196,41604,01/24/18,Trash - Curbside/Alley Missed Pickup,cart still out,NaN,37218.0,Red River,3203,1.0,1.715186e+06,682289.961678,-86.858594,36.204659
1646,48203,02/22/18,Trash - Backdoor,missed for 3 weeks 698 harding pl,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
2467,58920,04/06/18,Trash - Curbside/Alley Missed Pickup,missed- trash,NaN,37206.0,Metro,9503,5.0,1.747402e+06,674741.056063,-86.749210,36.184650
2508,59517,04/10/18,Trash - Curbside/Alley Missed Pickup,Trash was not pick up last week.,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
2760,62051,04/22/18,Trash - Curbside/Alley Missed Pickup,They forgot to pick up trash from apartment complex.,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN
9747,157096,03/20/19,Trash - Curbside/Alley Missed Pickup,Trash has not been picked up on the whole street.,NaN,NaN,No hauler identified,NaN,NaN,NaN,NaN,NaN,NaN


There are only nine items missing an address - I'm just going to discard them.

In [19]:
missed_pickups = missed_pickups.loc[-missed_pickups['Incident Address'].isna()].reset_index(drop = True)

In [20]:
missed_pickups.loc[
    missed_pickups['Incident Address'].str.contains('Nashville', case = False) & 
    ~missed_pickups['Incident Address'].str.contains('Nashville')]

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
1963,52415,03/08/18,Trash - Curbside/Alley Missed Pickup,Trash has not been picked up for 3 weeks,"1246 Antioch PIke, nashville tn 37211",37211.0,No hauler identified,NaN,13.0,1.763418e+06,640137.085506,-86.69411,36.08992


There is one instance of Nashville being mentioned in all lowercase; all other mentions of it are in title case ('Nashville')

In [21]:
for i, r in missed_pickups.iterrows():
    if ('Nashville' in r['Incident Address']) or ('nashville' in r['Incident Address']):
        ## looking for any characters preceding ' Nashville' or ' nashville', followed by any number of other characters
        missed_pickups.loc[i, 'Street Address'] = re.search(r'(.*),*\s?[nN]ashville.*', r['Incident Address']).group(1).replace(',', '').replace('.', '').strip().title()
    else:
        missed_pickups.loc[i, 'Street Address'] = r['Incident Address'].replace(',', '').replace('.', '').strip().title()

In [22]:
missed_pickups.sort_values('Street Address')

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE,Street Address
3964,79395,06/29/18,Trash - Curbside/Alley Missed Pickup,the entire street was missed,1 BELLE FORREST AVE C,37206.0,Metro,9502,7.0,1.751942e+06,677895.310490,-86.733905,36.193409,1 Belle Forrest Ave C
3997,79884,07/02/18,Trash - Curbside/Alley Missed Pickup,Missed entire street- carts are curbside in front of the home.,10 Belle Forrest Ave,37206.0,Metro,9502,7.0,1.751718e+06,678077.934103,-86.734668,36.193906,10 Belle Forrest Ave
9636,155122,03/15/19,Trash - Curbside/Alley Missed Pickup,MISS,"100 Bluefield Square, Nashville, TN 37214, United States",37214.0,Red River,1505,15.0,1.770431e+06,666861.601362,-86.670995,36.163465,100 Bluefield Square
1947,52252,03/07/18,Trash - Curbside/Alley Missed Pickup,Missed- trash,100 Braxton Hill Ct,37204.0,Red River,3302S,25.0,1.733781e+06,640909.303557,-86.794435,36.091422,100 Braxton Hill Ct
6808,121431,12/05/18,Trash - Curbside/Alley Missed Pickup,Missed- trash,100 Brook Hollow Rd,37205.0,Red River,1303,23.0,1.708043e+06,642454.642918,-86.881589,36.095062,100 Brook Hollow Rd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14225,224685,08/02/19,Trash - Curbside/Alley Missed Pickup,Trash is scheduled for collection on Thursday. Trash has not been collected for the entire street on both sides. All trash bins were at the road before 7:00am Thursday.,"Tusculum Rd, Nashville, TN , United States",37013.0,No hauler identified,NaN,30.0,1.762130e+06,627839.206313,-86.698171,36.056113,Tusculum Rd
14091,224234,08/01/19,Trash - Curbside/Alley Missed Pickup,Trash carts were placed at the road before 7:00 a.m. Thursday. It is now 9:00 pm and trash cart has not been picked up on the entire street.,"Tusculum Rd, Nashville, TN , United States",37013.0,No hauler identified,NaN,30.0,1.762130e+06,627839.206313,-86.698171,36.056113,Tusculum Rd
13495,217242,07/18/19,Trash - Curbside/Alley Missed Pickup,Trash was picked up on one side of the street and not the other. All trash bins were at the road before 7:00 a.m Thursday.,"Tusculum Rd, Nashville, TN , United States",37013.0,No hauler identified,NaN,30.0,1.762130e+06,627839.206313,-86.698171,36.056113,Tusculum Rd
15030,232129,08/15/19,Trash - Curbside/Alley Missed Pickup,miss,"Westboro Dr, Nashville, TN 37209, United States",37209.0,Red River,4203,20.0,1.710106e+06,662979.800016,-86.875229,36.151493,Westboro Dr


#### Check for duplicate compaints at the same address on the same day:

In [23]:
missed_pickups[missed_pickups.duplicated(['Date Opened', 'Street Address'], keep = False)].sort_values('Date Opened')

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE,Street Address
782,36149,01/02/18,Trash Collection Complaint,missed pick up - picked up everyone except her - wants to know why \n615-886-8389,631 e trinity ln,37207.0,Metro,9206,5.0,1.746385e+06,682160.779101,-86.752850,36.205010,631 E Trinity Ln
781,36148,01/02/18,Trash Collection Complaint,missed pick up - picked up everyone except her - wants to know why \n615-886-8389,631 e trinity ln,37207.0,No hauler identified,NaN,5.0,1.746385e+06,682160.779101,-86.752850,36.205010,631 E Trinity Ln
854,36996,01/04/18,Trash Collection Complaint,cust says trash was missed two weeks in a row,2405 Crestmoor Rd,37215.0,Red River,3305,25.0,1.726671e+06,647771.062641,-86.818690,36.110110,2405 Crestmoor Rd
853,36995,01/04/18,Trash - Curbside/Alley Missed Pickup,cust says missed pickup,2405 Crestmoor Rd,37215.0,Red River,3305,25.0,1.726671e+06,647771.062641,-86.818690,36.110110,2405 Crestmoor Rd
850,36957,01/04/18,Trash - Backdoor,Missed- Backdoor pickup,211 Robin Hill Rd,37205.0,Red River,1303,23.0,1.705095e+06,643896.868110,-86.891610,36.098950,211 Robin Hill Rd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
723,35407,12/27/17,Trash - Curbside/Alley Missed Pickup,2nd time this month All 12 condos have been missed.,101 Westover Park Ct,37215.0,Red River,3305S,25.0,1.729295e+06,644214.387862,-86.809710,36.100400,101 Westover Park Ct
724,35410,12/27/17,Trash - Curbside/Alley Missed Pickup,Missed .,101 Westover Park Ct,37215.0,Red River,3305S,25.0,1.729295e+06,644214.387862,-86.809710,36.100400,101 Westover Park Ct
679,35119,12/27/17,Trash - Curbside/Alley Missed Pickup,TRASH DIDNT GET PICKED UP ON FRIDAY,1754 Sprucedale Dr,37013.0,Red River,4510,32.0,1.787323e+06,621089.287156,-86.612799,36.038034,1754 Sprucedale Dr
7319,126555,12/29/18,Trash - Curbside/Alley Missed Pickup,All requirements were met,"206 Radnor St, Nashville, TN 37211, United States",37211.0,Red River,4304,16.0,1.750256e+06,645757.572128,-86.738798,36.105095,206 Radnor St


In [24]:
missed_pickups = missed_pickups.drop_duplicates(['Date Opened', 'Street Address']).reset_index(drop=True)

In [25]:
## beginning of input, any number of digits: r'^\d+'
missed_pickups['Building Number'] = missed_pickups['Street Address'].str.extract(r'(^\d+)')

In [26]:
missed_pickups.loc[missed_pickups['Building Number'].isna()].shape

(37, 15)

There are 37 additional rows that have no specific building listed in the address, just a street.  This is still negligible (.2% of rows) and will be discarded

In [27]:
missed_pickups = missed_pickups.loc[~missed_pickups['Building Number'].isna()].reset_index(drop = True)

In [28]:
missed_pickups['Street'] = missed_pickups['Street Address'].str.extract(r'(\b\D.*)')

In [29]:
missed_pickups

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE,Street Address,Building Number,Street
0,25274,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,Red River,4202,1.0,1.721259e+06,685444.799565,-86.838103,36.213470,4028 Clarksville Pike,4028,Clarksville Pike
1,25276,11/01/17,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,Red River,4205,20.0,1.707027e+06,659887.471571,-86.885562,36.142923,6528 Thunderbird Dr,6528,Thunderbird Dr
2,25307,11/01/17,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,Waste Ind,2206,2.0,1.735692e+06,685027.245923,-86.789170,36.212652,2603 Old Matthews Rd,2603,Old Matthews Rd
3,25312,11/01/17,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,Red River,4203,20.0,1.710186e+06,664205.101066,-86.874995,36.154861,604 Croley Dr,604,Croley Dr
4,25327,11/01/17,Trash Collection Complaint,"Trash out on time, miss again Tuesday. ALLEY",1816 Jo Johnston Ave,37203.0,Metro,9208,21.0,1.731459e+06,666013.601229,-86.802988,36.160330,1816 Jo Johnston Ave,1816,Jo Johnston Ave
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17738,267121,11/01/19,Trash - Curbside/Alley Missed Pickup,missed,"2709 Crestdale Dr, Nashville, TN 37214, United States",37214.0,Red River,1502,15.0,1.770240e+06,676334.399319,-86.671860,36.189483,2709 Crestdale Dr,2709,Crestdale Dr
17739,267125,11/01/19,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,Red River,4502,32.0,1.781137e+06,632448.551144,-86.633970,36.069130,2731 Murfreesboro Pike,2731,Murfreesboro Pike
17740,267126,11/01/19,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,Metro,9508,6.0,1.749711e+06,669201.601569,-86.741242,36.169482,1621 Long Ave,1621,Long Ave
17741,267130,11/01/19,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, United States",37214.0,Red River,1502,15.0,1.770293e+06,674936.303809,-86.671647,36.185643,2943 Windemere Cir,2943,Windemere Cir


#### Some trash routes are not purely numeric - they have an 'S' at the end, which feels intentional.  If we do any analysis by route, we might want to look into that.

In [30]:
missed_pickups['Trash Route'].value_counts(dropna = False)

Trash Route
NaN      725
4504     325
3302     289
4404     252
9303     248
        ... 
2405S      3
3303S      2
2505S      2
4201S      2
1502S      1
Name: count, Length: 173, dtype: int64

In [31]:
missed_pickups.sort_values('Trash Route')['Trash Route'].unique()

array(['1201', '1202', '1202S', '1203', '1204', '1205', '1301', '1302',
       '1302S', '1303', '1303S', '1304', '1305', '1308', '1309', '1401',
       '1402', '1403', '1404', '1405', '1501', '1502', '1502S', '1503',
       '1504', '1504S', '1505', '1507', '2201', '2202', '2203', '2204',
       '2205', '2206', '2207', '2301', '2301S', '2302', '2303', '2303S',
       '2304', '2304S', '2305', '2305S', '2306', '2308', '2401', '2402',
       '2402S', '2403', '2404', '2405', '2405S', '2408', '2501', '2502',
       '2503', '2504', '2505', '2505S', '2506', '2509', '3201', '3202',
       '3203', '3204', '3205', '3206', '3207', '3208', '3212', '3214',
       '3301', '3301S', '3302', '3302S', '3303', '3303S', '3304', '3304S',
       '3305', '3305S', '3306', '3307', '3312', '3314', '3401', '3402',
       '3402S', '3403', '3404', '3405', '3406', '3407', '3411', '3412',
       '3414', '3501', '3502', '3503', '3504', '3505', '3512', '3514',
       '4201', '4201S', '4202', '4203', '4203S', '4204', '4

### Top 5 addresses with missed pickups

In [34]:
addresses = missed_pickups.groupby(['Street Address', 'Zip Code'], as_index = False).agg(missed = ('Request Number', 'count')).sort_values('missed', ascending = False)

In [35]:
addresses['Zip Code'] = addresses['Zip Code'].astype(int)

In [36]:
addresses.head(5)

,Street Address,Zip Code,missed
568,110 George L Davis Blvd,37203,26
8871,5135 Hickory Hollow Pkwy,37013,23
1240,12546 Old Hickory Blvd,37013,20
6658,3710 N Natchez Ct,37211,20
2017,1584 Bell Rd,37211,19


In [40]:
addresses.describe()

,Zip Code,missed
count,11598.000000,11598.000000
mean,37179.632264,1.525435
std,67.512581,1.295517
min,37013.000000,1.000000
25%,37205.000000,1.000000
50%,37209.000000,1.000000
75%,37214.000000,2.000000
max,37228.000000,26.000000


In [37]:
streets = missed_pickups.groupby(['Street', 'Zip Code'], as_index = False).agg(missed = ('Request Number', 'count')).sort_values('missed', ascending = False)

In [38]:
streets['Zip Code'] = streets['Zip Code'].astype(int)

In [39]:
streets.head(5)

,Street,Zip Code,missed
675,Buena Vista Pike,37218,91
1656,Granny White Pike,37204,70
2798,Old Hickory Blvd,37013,67
1298,Eastland Ave,37206,61
268,Annex Ave,37209,59


## Determine damages due to missed pickups
* The first missed pickup at an address will not result in a fine
* Every subsequent missed pickup will result in a $200 fine

In [ ]:
missed_pickups['Trash Hauler'].value_counts()

For each trash hauler, we need:  
* A: A list of each unique addresses with missed pickup
* B: The number of unique addresses with missed pickups
* C: The number of times that each address was missed
  
Then the fine should be 200 * (SUM(C) - B)

In [ ]:
unique_haulers = missed_pickups['Trash Hauler'].unique()

In [ ]:
m1_damages = []

for hauler in unique_haulers:
    missed = missed_pickups.loc[missed_pickups['Trash Hauler'] == hauler]

    rows = missed.shape[0]
    addresses = missed['Street Address'].nunique()
    fine = 200 * (rows - addresses)

    m1_damages.append({'hauler': hauler, 'missed pickups': rows, 'Initial': fine})

m1_damages = pd.DataFrame(m1_damages)
m1_damages

In [ ]:
m1_damages = m1_damages.sort_values('Initial')

In [ ]:
fig, ax = plt.subplots(figsize = (4, 6))

ax.barh(y=m1_damages['hauler'], width = (m1_damages['Initial']/100000), color = 'wheat')

ax.get_children()[3].set_color('chocolate')

ax.spines[['right', 'top']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('gray')
ax.tick_params(colors = 'black')
ax.set_xticks(range(0,11))

ax.set_xlabel('Fine (in $100k)', color = 'dimgray')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (4, 6))

ax.barh(y=m1_damages['hauler'], width = (m1_damages['missed pickups']/1000), color = 'wheat')

ax.get_children()[2].set_color('chocolate')

ax.spines[['right', 'top']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('gray')
ax.tick_params(colors = 'black')

ax.set_xlabel('Number of missed pickups (thousands)', color = 'dimgray')

plt.title('Missed Pickups by Trash Hauler')

plt.show()

## What other kinds of complaints are there?

In [ ]:
other_complaints = data_load.loc[
    ~(data_load['Request '].str.contains('Missed')) & 
    ## any capitalization of 'miss'
    ~(data_load['Description'].str.contains(r'\b[mM][iI][sS]{2}.?', regex = True)) & 
    ## any capitalization of 'no pickup'/'no pick up'/'not picked up'
    ~(data_load['Description'].str.contains(r'\b[nN][oO].*\b[pP][iI][cC][kK].*[uU][pP]', regex = True))].reset_index(drop = True)

In [ ]:
other_complaints.head(50)

In [ ]:
complaints = other_complaints['Description'].str.lower()

In [ ]:
word_list = []
for complaint in complaints:
    words = complaint.split()
    for word in words:
        word_list.append(word)

In [ ]:
ignore = ['picked', 'our', 'needs', 'when', 'down', 'there', 'get', 'would', 'the', 'and', 'trash', 'to', 'in', 'of', 'is', 'a', 'on', 'it', 'this', 'that', 'not', 'was', 'up', 'out', 'has', 'they', 'her', 'i', 'he', 'are', 'my', 'she', 'for', 'have', 'carts', 'all', 'be', 'at', 'over', 'from', 'says', 'caller', 'wants', 'his', 'been', 'with', 'can', 'states', 'call', 'still', 'but']

In [ ]:
word_list = pd.Series(word_list)

In [ ]:
word_list[~word_list.isin(ignore)].value_counts().head(50)

## Geospatial Analysis
* Are there any geospatial analysis you can do?  Which visualizations can you create?

(Will use a different notebook for this, just going to export my missed_pickups here)

In [ ]:
missed_pickups.to_csv('../data/missed_pickups_rme.csv')

## Answers
* __Total damages owed by Red River Waste Solutions due to missed pickups:__ \$861,200
* __How much does each trash hauler owe?__
  * Red River: \$861,200
  * Waste Industries: \$72,800
  * Metro: \$207,600
  * There is an additional \$13,200 on the table due to haulers not being connected to specific addresses
* __How do metro crews compare to the contractor's performance?__
  * Metro crews missed less than 1/4 of the pickups that Red River missed, but without knowing the total number of pickups in each crew's area, we can't compare rates of misses. 

## Bonus

### Alternative Method 1:

For each address, if there are three or more missed pickups within a 180-day period, damages of $1500 will be charged. (A fine will be levied every time three unique missed pickup dates occur within a six-month period for a single address.)

for each address  
    for each date  
        are there 2 dates within 180 days following the initial date?
    

In [ ]:
missed_pickups['Date Opened'] = pd.to_datetime(missed_pickups['Date Opened'], format = '%m/%d/%y')

In [ ]:
a1_damages = []

for hauler in unique_haulers:
    damages = 0
    unique_addresses = missed_pickups.loc[missed_pickups['Trash Hauler'] == hauler, 'Street Address'].unique()
    
    for address in unique_addresses:
        missed_dates = missed_pickups.loc[(missed_pickups['Street Address'] == address) & (missed_pickups['Trash Hauler'] == hauler), 'Date Opened']

        for date in missed_dates:
            range_end = date + timedelta(days=180)
            misses_in_range = len(missed_dates.loc[lambda x : (x >= date) & (x <= range_end)])

            if misses_in_range >= 3:
                damages += 1500
        
    a1_damages.append({'hauler': hauler, 'Alternative 1': damages})

a1_damages = pd.DataFrame(a1_damages)
a1_damages

### Alternative Method 2:

This method also considers the six-month window like Alternative Method 1, but each date can only be used once to support a fine. How will this difference impact the fines levied? 

for each hauler  
for each address  
for each date  
add it to a list  
while list length < 3:
how far away is the next closest date?  If < 180, add to the list 

In [ ]:
a2_damages = []

for hauler in unique_haulers:
    unique_addresses = missed_pickups.loc[missed_pickups['Trash Hauler'] == hauler, 'Street Address'].unique()
    damages = 0

    for address in unique_addresses:
        missed_dates = missed_pickups.sort_values('Date Opened').loc[(missed_pickups['Street Address'] == address) & (missed_pickups['Trash Hauler'] == hauler), 'Date Opened'].to_list()
        i = 0

        while i+3 < len(missed_dates):
            first_date = missed_dates[i]
            next_date = missed_dates[i + 1]
            third_date = missed_dates[i + 2]
    
            if (third_date - first_date) <= timedelta(days = 180):
                damages += 1500
                i += 3
            else: 
                i += 1

    a2_damages.append({'hauler': hauler, 'Alternative 2': damages})

a2_damages = pd.DataFrame(a2_damages)
a2_damages

### Combining all methods

In [ ]:
merge1 = pd.merge(m1_damages, a1_damages, how = 'left', on = 'hauler')

In [ ]:
all_methods = pd.merge(merge1, a2_damages, how = 'left', on = 'hauler').sort_values('missed pickups', ascending = False)

In [ ]:
all_methods

In [ ]:
fig, ax = plt.subplots(figsize = (6,4))

bar_width = 0.25
bar1 = [0, 1, 2, 3]
bar2 = [x + bar_width for x in bar1]
bar3 = [x + bar_width for x in bar2]

ax.bar(bar1, all_methods['Initial']/100000, color = 'wheat', width = bar_width, label = 'Initial')
ax.bar(bar2, all_methods['Alternative 1']/100000, color = 'peru', width = bar_width, label = 'Alternative 1')
ax.bar(bar3, all_methods['Alternative 2']/100000, color = 'saddlebrown', width = bar_width, label = 'Alternative 2')

ax.set_xticks(bar2)
ax.set_xticklabels(all_methods['hauler'])

ax.set_yticks(range(0, 25, 5))
ax.set_ylabel('Potential Fine ($100k)', color = 'dimgray')

ax.spines[['right', 'top']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('gray')
ax.tick_params(colors = 'black')

plt.title('Comparison of Fine Methods')

ax.legend()

plt.show();